# Task 1: Traffic Speed Prediction — XGBoost Submission

Predict traffic speed (km/h) for 1,260 road segments at +20, +40, and +60 minutes into the future.

**Approach**: Global XGBoost model with handcrafted features from speed history, road network adjacency, and road metadata.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import xgboost as xgb

from src import (
    load_train_speeds, load_train_texts, load_test_data,
    load_adjacency, load_roads,
    build_windows, build_features,
    write_submission, validate_submission,
)

## 1. Load & Prepare Data

Load both training blocks (16,199 total timesteps) and generate supervised sliding windows. Each window contains 15 steps of speed history (1 hour) and targets at +5, +10, +15 steps ahead.

In [ ]:
s1, s2 = load_train_speeds()
t1, t2 = load_train_texts()
adj = load_adjacency()
roads = load_roads()

X1, T1, Y1 = build_windows(s1, t1)
X2, T2, Y2 = build_windows(s2, t2)

X_all = np.concatenate([X1, X2], axis=0).astype(np.float32)
Y_all = np.concatenate([Y1, Y2], axis=0).astype(np.float32)

print(f"Train samples: {len(X_all)}")

## 2. Feature Engineering

For each road in each window, extract 12 features:
- **Lag speeds** at t-0, t-1, t-3, t-7, t-14 (captures recent trends)
- **Statistics**: mean, standard deviation, and trend (last − first) over the 15-step window
- **Road metadata**: road class and segment length
- **Neighbor speeds**: mean speed of adjacent roads (from adjacency matrix) at last step and 3 steps back

Features are built in a vectorized pass — no Python loops over roads.

In [ ]:
print("Building features...")
F_all = build_features(X_all, adj, roads)
Y_flat = Y_all.transpose(0, 2, 1).reshape(-1, 3)
print(f"Features: {F_all.shape}, Targets: {Y_flat.shape}")

## 3. Train XGBoost Models

Train one XGBoost regressor per horizon (h5, h10, h15). Each model learns a mapping from the 12 engineered features to the future speed at that horizon. All roads share the same model — the road identity is encoded via metadata and neighbor features.

In [ ]:
models = {}
for hi, hname in enumerate(["h5", "h10", "h15"]):
    model = xgb.XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, verbosity=0, n_jobs=-1,
    )
    model.fit(F_all, Y_flat[:, hi])
    models[hname] = model
    print(f"  {hname} done")

## 4. Predict on Test Set

The test set contains 540 fixed history windows (no targets). Build the same features, predict all three horizons per road, and clip negative predictions to zero (speed cannot be negative).

In [ ]:
test_hist, _ = load_test_data()
F_test = build_features(test_hist, adj, roads)
print(f"Test features: {F_test.shape}")

test_preds = np.zeros((540, 3, 1260), dtype=np.float32)
for hi, hname in enumerate(["h5", "h10", "h15"]):
    test_preds[:, hi, :] = models[hname].predict(F_test).reshape(540, 1260)
test_preds = test_preds.clip(min=0)

print(f"Predictions: {test_preds.shape}, range=[{test_preds.min():.1f}, {test_preds.max():.1f}]")

## 5. Write Submission

Generate `submission.csv` with 2,041,200 rows (540 samples × 3 horizons × 1,260 roads), matching the exact `id,speed` format from `sample_submission.csv`. The submission is validated to ensure all row IDs match in count and order.

In [ ]:
out_dir = write_submission(test_preds, label="xgb_final", models=models)
validate_submission(out_dir / "submission.csv")